# Europlátano Accuracy--Coherence Plot

Este notebook extrae métricas de:
- `2.Production-KAN vs KAN-MM.txt`
- `2.Production-TabNet.txt`

y genera una figura para LaTeX con:
- X-axis: MR violation rate (%)
- Y-axis: RMSE (raw units)
- Puntos: TabNet, KAN, KAN+MR

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_dir = Path('/Users/oroncal/workspace/projects/picota/runtime.test/analysis/europlatano')
kan_vs_mm_log = analysis_dir / '2.Production-KAN vs KAN-MM.txt'
tabnet_log = analysis_dir / '2.Production-TabNet.txt'

text_kan_mm = kan_vs_mm_log.read_text(encoding='utf-8')
text_tabnet = tabnet_log.read_text(encoding='utf-8')

print(f'Loaded: {kan_vs_mm_log}')
print(f'Loaded: {tabnet_log}')

Loaded: /Users/oroncal/workspace/projects/picota/runtime.test/analysis/europlatano/2.Production-KAN vs KAN-MM.txt
Loaded: /Users/oroncal/workspace/projects/picota/runtime.test/analysis/europlatano/2.Production-TabNet.txt


In [2]:
def parse_model_metrics(text: str, model_label: str):
    m = re.search(
        rf"{re.escape(model_label)} test: .*?mae_raw=([0-9.]+) rmse_raw=([0-9.]+)",
        text,
    )
    if not m:
        raise ValueError(f'No test metrics found for {model_label}')
    mae_raw = float(m.group(1))
    rmse_raw = float(m.group(2))
    return mae_raw, rmse_raw


def parse_violation_rate(text: str, model_label: str):
    m = re.search(
        rf"rule_violations\\[{re.escape(model_label)}:test\\]: overall_rate=([0-9.]+)",
        text,
    )
    if not m:
        raise ValueError(f'No violation rate found for {model_label}')
    return float(m.group(1))


metrics = {
    'KAN': {},
    'KAN-Mm': {},
    'TabNet': {},
}

metrics['KAN']['mae_raw'], metrics['KAN']['rmse_raw'] = parse_model_metrics(text_kan_mm, 'KAN')
metrics['KAN']['violation_rate'] = parse_violation_rate(text_kan_mm, 'KAN')
metrics['KAN']['lambda'] = 0.0

metrics['KAN-Mm']['mae_raw'], metrics['KAN-Mm']['rmse_raw'] = parse_model_metrics(text_kan_mm, 'KAN-Mm')
metrics['KAN-Mm']['violation_rate'] = parse_violation_rate(text_kan_mm, 'KAN-Mm')
metrics['KAN-Mm']['lambda'] = 0.25

metrics['TabNet']['mae_raw'], metrics['TabNet']['rmse_raw'] = parse_model_metrics(text_tabnet, 'tabnet')
metrics['TabNet']['violation_rate'] = parse_violation_rate(text_tabnet, 'tabnet')
metrics['TabNet']['lambda'] = np.nan

rows = []
for model, values in metrics.items():
    rows.append({
        'model': model,
        'lambda': values['lambda'],
        'mae_raw': values['mae_raw'],
        'rmse_raw': values['rmse_raw'],
        'violation_rate_pct': values['violation_rate'] * 100.0,
    })

df = pd.DataFrame(rows)
df

ValueError: No violation rate found for KAN

In [ ]:
plot_df = df.copy()
plot_df['model_plot'] = plot_df['model'].replace({'KAN-Mm': 'KAN+MR'})
plot_df = plot_df[['model_plot', 'violation_rate_pct', 'rmse_raw']]

model_order = ['TabNet', 'KAN', 'KAN+MR']
plot_df = plot_df.set_index('model_plot').loc[model_order].reset_index()

model_colors = {
    'TabNet': '#2ca02c',
    'KAN': '#1f77b4',
    'KAN+MR': '#d62728',
}

fig, ax = plt.subplots(figsize=(7.2, 4.6))

# Connected scatter (trade-off path)
ax.plot(
    plot_df['violation_rate_pct'],
    plot_df['rmse_raw'],
    color='#6b6b6b',
    linewidth=1.4,
    alpha=0.75,
    zorder=1,
)

for i in range(len(plot_df) - 1):
    x0 = float(plot_df.loc[i, 'violation_rate_pct'])
    y0 = float(plot_df.loc[i, 'rmse_raw'])
    x1 = float(plot_df.loc[i + 1, 'violation_rate_pct'])
    y1 = float(plot_df.loc[i + 1, 'rmse_raw'])
    ax.annotate(
        '',
        xy=(x1, y1),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle='->', color='#6b6b6b', lw=1.4, shrinkA=10, shrinkB=10),
    )

for _, row in plot_df.iterrows():
    model = row['model_plot']
    x = float(row['violation_rate_pct'])
    y = float(row['rmse_raw'])
    ax.scatter(
        x,
        y,
        s=85,
        color=model_colors[model],
        edgecolor='white',
        linewidth=0.8,
        zorder=3,
        label=model,
    )
    ax.annotate(
        model,
        (x, y),
        textcoords='offset points',
        xytext=(6, 6),
        fontsize=9,
    )

# Indicate preferred region: lower-left (less violations, less error).
x_min, x_max = float(plot_df['violation_rate_pct'].min()), float(plot_df['violation_rate_pct'].max())
y_min, y_max = float(plot_df['rmse_raw'].min()), float(plot_df['rmse_raw'].max())
x_span = max(1e-6, x_max - x_min)
y_span = max(1e-6, y_max - y_min)
ax.annotate(
    'Mejor',
    xy=(x_min + 0.03 * x_span, y_min + 0.03 * y_span),
    xytext=(x_min + 0.28 * x_span, y_min + 0.30 * y_span),
    arrowprops=dict(arrowstyle='->', color='#333333', lw=1.2),
    fontsize=9,
    color='#333333',
)

ax.set_title('Trade-off accuracy--coherence for Europlátano', fontsize=12)
ax.set_xlabel('MR violation rate (%)')
ax.set_ylabel('RMSE (raw units)')
ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.35)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:3], labels[:3], loc='best', frameon=False)

fig.tight_layout()

out_pdf = analysis_dir / 'accuracy_coherence_europlatano.pdf'
out_png = analysis_dir / 'accuracy_coherence_europlatano.png'
fig.savefig(out_pdf, bbox_inches='tight')
fig.savefig(out_png, dpi=300, bbox_inches='tight')

print('Saved figure:')
print(' -', out_pdf)
print(' -', out_png)
plt.show()